# Week 65, Day 1: Hidden Markov models: filtering, smoothing and the best path

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w065/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson
g0 = np.random.default_rng(651)
T = 300
A = np.array([[0.96, 0.04], [0.03, 0.97]])
rates = np.array([2.0, 3.6])
z = np.zeros(T, int)
for t in range(1, T):
    z[t] = g0.choice(2, p=A[z[t - 1]])
x = g0.poisson(rates[z])
B = poisson.pmf(x[:, None], rates)                      # T x 2 emission likelihoods
print(f"{np.mean(z == 1):.0%} of bins in the up state; mean count {x[z == 0].mean():.1f} (down) and {x[z == 1].mean():.1f} (up)")

### Exercise: Forward-backward with scaling

Write forward(B, A, pi) returning the normalised forward messages and the scaling factors, and backward(B, A, c) returning the scaled backward messages. Compare the accuracy of filtered and smoothed state estimates.

In [ ]:
def forward(B, A, pi):
    T_, K = B.shape
    alpha = np.zeros((T_, K)); c = np.zeros(T_)
    a = pi * B[0]; c[0] = a.sum(); alpha[0] = a / c[0]
    for t in range(1, T_):
        a = ...
        c[t] = a.sum(); alpha[t] = a / c[t]
    return alpha, c

def backward(B, A, c):
    T_, K = B.shape
    beta = np.ones((T_, K))
    for t in range(T_ - 2, -1, -1):
        beta[t] = ...
    return beta

pi0 = np.array([0.5, 0.5])
alpha, c = forward(B, A, pi0)
beta = backward(B, A, c)
filt = alpha
smooth = alpha * beta
smooth /= smooth.sum(1, keepdims=True)
acc_filt = np.mean((filt[:, 1] > 0.5) == (z == 1))
acc_smooth = np.mean((smooth[:, 1] > 0.5) == (z == 1))
print(f"log-likelihood {np.sum(np.log(c)):.2f}; accuracy filtered {acc_filt:.3f}, smoothed {acc_smooth:.3f}")

In [ ]:
import numpy as np
def test_norm():
    "filtered and smoothed posteriors are normalised at each time"
    assert np.allclose(filt.sum(1), 1) and np.allclose(smooth.sum(1), 1)
def test_loglik():
    "the log-likelihood from the scaling factors matches a brute-force sum over the first 10 steps"
    import itertools
    tot = 0.0
    for path in itertools.product((0, 1), repeat=10):
        p = 0.5 * B[0, path[0]]
        for t in range(1, 10):
            p *= A[path[t - 1], path[t]] * B[t, path[t]]
        tot += p
    alpha10, c10 = forward(B[:10], A, np.array([0.5, 0.5]))
    assert abs(np.sum(np.log(c10)) - np.log(tot)) < 1e-10
def test_smooth_better():
    "smoothing recovers the state better than filtering"
    assert acc_smooth > acc_filt

test_norm()
test_loglik()
test_smooth_better()
print("All checks passed")

### Exercise: The most likely path

Write viterbi(B, A, pi) using log-probabilities and backpointers, and logp(path), the log joint probability of a state path with the observations. Compare the Viterbi path with pointwise smoothed decoding and with thresholding the counts at 2.8.

In [ ]:
def viterbi(B, A, pi):
    T_, K = B.shape
    lB, lA = np.log(B + 1e-300), np.log(A)
    d = np.zeros((T_, K)); bp = np.zeros((T_, K), int)
    d[0] = np.log(pi) + lB[0]
    for t in range(1, T_):
        cand = d[t - 1][:, None] + lA                   # K x K: from j to k
        bp[t] = np.argmax(cand, axis=0)
        d[t] = ...
    path = np.zeros(T_, int); path[-1] = int(np.argmax(d[-1]))
    for t in range(T_ - 2, -1, -1):
        path[t] = bp[t + 1, path[t + 1]]
    return path

def logp(path):
    out = np.log(pi0[path[0]]) + np.log(B[0, path[0]] + 1e-300)
    for t in range(1, len(path)):
        out += np.log(A[path[t - 1], path[t]]) + np.log(B[t, path[t]] + 1e-300)
    return out

switches = lambda p: int(np.sum(p[1:] != p[:-1]))
path_vit = viterbi(B, A, pi0)
path_point = (smooth[:, 1] > 0.5).astype(int)
path_thresh = (x > 2.8).astype(int)
for name, p in [("Viterbi", path_vit), ("pointwise", path_point), ("threshold", path_thresh), ("truth", z)]:
    print(f"{name:10s} accuracy {np.mean(p == z):.3f}, switches {switches(p):3d}, log joint {logp(p):9.2f}")

In [ ]:
import numpy as np
def test_path():
    "the Viterbi path is at least as likely as the smoothed pointwise path"
    assert logp(path_vit) >= logp(path_point) - 1e-9
def test_switches():
    "it switches far less often than thresholding the counts"
    assert switches(path_vit) < switches(path_thresh)

test_path()
test_switches()
print("All checks passed")

### Exercise: An impossible pointwise path

Build a three-state chain in which state 0 cannot be followed by state 2, with observations that nevertheless favour 0 then 2. Compare the pointwise and Viterbi decodings.

In [ ]:
A3 = np.array([[0.5, 0.5, 0.0], [0.3, 0.4, 0.3], [0.0, 0.5, 0.5]])
pi3 = np.array([1 / 3, 1 / 3, 1 / 3])
B3 = np.array([[0.7, 0.2, 0.1], [0.3, 0.1, 0.6], [0.25, 0.1, 0.65]])      # 3 time steps, 3 states
a3, c3 = forward(B3, A3, pi3)
b3 = backward(B3, A3, c3)
sm3 = a3 * b3; sm3 /= sm3.sum(1, keepdims=True)
p_point3 = ...
p_vit3 = viterbi(B3, A3, pi3)

def logp3(path):
    out = np.log(pi3[path[0]]) + np.log(B3[0, path[0]])
    for t in range(1, len(path)):
        out += np.log(A3[path[t - 1], path[t]] + 0) + np.log(B3[t, path[t]])
    return out

print("smoothed posteriors:\n", np.round(sm3, 3))
print(f"pointwise path {p_point3}, log probability {logp3(p_point3)}")
print(f"Viterbi path   {p_vit3}, log probability {logp3(p_vit3):.3f}")

In [ ]:
import numpy as np
def test_impossible():
    "the pointwise path has probability zero under the model, while the Viterbi path does not"
    assert np.isneginf(logp3(p_point3)) and np.isfinite(logp3(p_vit3))

test_impossible()
print("All checks passed")